In [87]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import random
import oracledb
import cx_Oracle

<h2> 선호 카테고리 테이블 DB에 임시 저장</h2>
<p>select * from preference</p>

In [91]:
pref = pd.read_csv("C:\\Users\\PC\\Desktop\\library_test\\test2_preference.csv", index_col='mem_id', encoding='cp949')
pref

,cate_id
mem_id,
1,IT/과학
1,사회
1,경제
2,연예
2,사회
...,...
249,경제
249,생활/문화
250,사회


In [92]:
# 각 회원 별 무작위 선호 카테고리 난수 지정 -> 테스트 회원은 250명으로 임시 설정
arr = []
for i in range(250):
    tmp = random.sample(range(7), 3)
    arr.append(tmp[0])
    arr.append(tmp[1])
    arr.append(tmp[2])

print(arr)

[0, 4, 3, 1, 3, 4, 6, 4, 3, 0, 6, 5, 0, 4, 5, 5, 2, 4, 5, 2, 4, 3, 6, 0, 3, 6, 1, 0, 4, 1, 0, 5, 6, 5, 2, 3, 4, 6, 0, 4, 6, 0, 5, 1, 4, 2, 5, 1, 6, 3, 2, 1, 5, 0, 3, 6, 2, 0, 6, 5, 4, 1, 0, 6, 3, 4, 5, 3, 2, 6, 4, 2, 3, 5, 1, 5, 3, 1, 1, 2, 4, 2, 0, 6, 3, 4, 2, 3, 4, 6, 4, 5, 6, 0, 1, 6, 2, 4, 3, 3, 6, 1, 0, 5, 2, 2, 3, 0, 5, 2, 3, 1, 5, 3, 5, 6, 2, 2, 0, 1, 0, 2, 4, 4, 1, 3, 3, 6, 0, 0, 5, 3, 3, 6, 5, 1, 4, 6, 0, 3, 1, 0, 3, 2, 4, 6, 5, 4, 1, 0, 3, 1, 2, 1, 2, 3, 3, 0, 5, 1, 6, 5, 1, 6, 3, 3, 0, 1, 5, 1, 4, 1, 0, 2, 0, 1, 6, 1, 3, 6, 1, 5, 0, 6, 5, 0, 2, 1, 5, 0, 6, 3, 1, 6, 3, 1, 0, 6, 3, 5, 0, 1, 0, 6, 3, 1, 5, 6, 3, 4, 1, 6, 0, 3, 0, 1, 0, 6, 3, 3, 6, 0, 0, 1, 6, 2, 0, 3, 3, 1, 5, 6, 1, 0, 5, 3, 4, 3, 2, 6, 0, 1, 6, 1, 6, 5, 0, 6, 1, 6, 1, 5, 5, 4, 6, 3, 1, 4, 1, 3, 0, 3, 2, 0, 2, 4, 6, 3, 6, 4, 6, 0, 2, 6, 3, 5, 2, 0, 6, 1, 6, 3, 4, 0, 1, 1, 0, 5, 1, 0, 4, 3, 1, 5, 3, 0, 1, 5, 1, 4, 3, 5, 0, 4, 3, 6, 1, 5, 0, 0, 6, 2, 0, 3, 5, 1, 3, 2, 2, 6, 0, 1, 2, 4, 3, 1, 2, 5, 1, 3, 4, 3, 2, 

In [93]:
# arr에 배정된 숫자에 따라 cate에서 인덱스로 카테고리 지정
cate = ['정치', '연예', '경제', '사회', '생활/문화', '세계', 'IT/과학']
pref['cate_id'] = [cate[a] for a in arr]
pref

,cate_id
mem_id,
1,정치
1,생활/문화
1,사회
2,연예
2,사회
...,...
249,정치
249,생활/문화
250,정치


In [94]:
pref.to_csv("C:\\Users\\PC\\Desktop\\library_test\\test2_preference.csv", encoding='cp949')
# 파일로 생성하는 코드

<h2>각 회원의 조회 기사를 각 회원의 카테고리 내에서 선택해서 배정</h2>
<p>select * from user_log(?)</p>

In [95]:
news = pd.read_csv("C:\\Users\\PC\\Desktop\\library_test\\test2_news.csv", index_col='news_id', encoding='cp949')
news
# 기사 제목, 아이디, 카테고리만 가져옴

,cate_id,title
news_id,,
0,정치,이탈리아 지지표 흡수는 기본… 중립 성향 표심 잡는 게 관건
1,정치,이준석 “영남 정치인들 편하게 놔두지 않겠다”…영남 신당 추진 시사
2,정치,주호영 “서울 갈 일 없다”…이준석 “대구에서 어려운 승부할 것”
3,정치,"28억 재산 누락' 김대기 ""단순 실수""…야당 ""대국민 사과해야"""
4,정치,"與 핵심, '거취 압박' 대응 자제...내부선 불만 기류도"
...,...,...
1389,연예,"김영,'훈훈한 미소'"
1390,연예,"신연서,'소중한 강아지'"
1391,연예,"최훈재,'나도 찰칵'"


In [97]:
# 250명이 자신의 카테고리에 소속된 기사 중 몇개를 무작위로 고르는 로그 16000개
data2 = {'mem_id' : [], 'news_id' : []}

while len(data2['mem_id']) < 16000:  # 로그가 16000개가 쌓일때까지 반복
    mem_id = np.random.randint(1,251)  # mem_id 랜덤 생성
    news_id = np.random.randint(0, 1394) # news_id 랜덤 생성

    sel_cate = pref[pref.index == mem_id]['cate_id'].values # mem_id 유저가 선택한 카테고리

    if news.loc[news_id, 'cate_id'] in sel_cate:
        data2['mem_id'].append(mem_id)
        data2['news_id'].append(news_id)

log2 = pd.DataFrame(data2)
log2

,mem_id,news_id
0,185,796
1,190,1239
2,63,1382
3,174,1214
4,175,477
...,...,...
15995,143,1144
15996,179,1175
15997,239,938
15998,125,1206


In [116]:
# 제대로 동작하는지 확인
user = 2  # 유저 번호 입력
article = 938 # 기사 번호 입력

print("입력한 유저가 고른 카테고리")
print(pref[pref.index == user].values)
print("==============================")

print("입력한 기사번호 소속 카테고리 및 제목")
print(news[news.index == article])

입력한 유저가 고른 카테고리
[[1 '사회']]
입력한 기사번호 소속 카테고리 및 제목
        cate_id                          title
news_id                                       
938          세계  “팔레스타인 가자지구가 어린이들의 무덤이 되고 있다”


In [106]:
log2_2 = pd.merge(log2, news, on='news_id') # 카테고리와 제목 가져오기
log2_2.to_csv("C:\\Users\\PC\\Desktop\\library_test\\test4_log2.csv", encoding='cp949')

여기서부터 DB 연결

In [107]:
log2_2

,mem_id,news_id,cate_id,title
0,185,796,생활/문화,겨울 시작 '입동' 추위…주말에는 더 강한 찬 바람
1,57,796,생활/문화,겨울 시작 '입동' 추위…주말에는 더 강한 찬 바람
2,247,796,생활/문화,겨울 시작 '입동' 추위…주말에는 더 강한 찬 바람
3,6,796,생활/문화,겨울 시작 '입동' 추위…주말에는 더 강한 찬 바람
4,57,796,생활/문화,겨울 시작 '입동' 추위…주말에는 더 강한 찬 바람
...,...,...,...,...
15995,12,209,경제,"20억 빌라 소유' 유해진, 성북동 150평 주택 샀다…45억 현찰"
15996,63,209,경제,"20억 빌라 소유' 유해진, 성북동 150평 주택 샀다…45억 현찰"
15997,113,209,경제,"20억 빌라 소유' 유해진, 성북동 150평 주택 샀다…45억 현찰"
15998,34,1129,IT/과학,‘통합’다나와·에누리 어떨까?…커넥트웨이브“AI로 고객 문제해결”(종합)


In [108]:
# DB 접속
con = cx_Oracle.connect(user="ist", password='ist', dsn='localhost:1521/xe')
cursor = con.cursor()

In [117]:
# 임의 난수 로그 insert
for i in range(16000):
    mid, nid, cid = log2_2.loc[i][:3]
    sql = "INSERT INTO user_log(log_id, mem_id, cate_id, news_id) VALUES (log_seq.NEXTVAL, '" + str(mid) + "', '" + str(cid) + "', '" + str(nid) +"')"
    cursor.execute(sql)
    con.commit()

In [110]:
pref = pd.read_csv("C:\\Users\\PC\\Desktop\\library_test\\test2_preference.csv", encoding='cp949')
pref

,mem_id,cate_id
0,1,정치
1,1,생활/문화
2,1,사회
3,2,연예
4,2,사회
...,...,...
745,249,정치
746,249,생활/문화
747,250,정치
748,250,사회


In [111]:
# 유저별 선택한 카테고리 insert
for i in range(750):
    mid, cid = pref.loc[i][:2]
    sql = "INSERT INTO test_pref(cate_id, mem_id) VALUES ('" + str(cid) + "', '" + str(mid) + "')"
    cursor.execute(sql)
    con.commit()

In [119]:
# 뉴스 DB에 삽입
news = pd.read_csv("C:\\Users\\PC\\Desktop\\library_test\\test2_news.csv", encoding='cp949')
news

,news_id,cate_id,title
0,0,정치,이탈리아 지지표 흡수는 기본… 중립 성향 표심 잡는 게 관건
1,1,정치,이준석 “영남 정치인들 편하게 놔두지 않겠다”…영남 신당 추진 시사
2,2,정치,주호영 “서울 갈 일 없다”…이준석 “대구에서 어려운 승부할 것”
3,3,정치,"28억 재산 누락' 김대기 ""단순 실수""…야당 ""대국민 사과해야"""
4,4,정치,"與 핵심, '거취 압박' 대응 자제...내부선 불만 기류도"
...,...,...,...
1389,1389,연예,"김영,'훈훈한 미소'"
1390,1390,연예,"신연서,'소중한 강아지'"
1391,1391,연예,"최훈재,'나도 찰칵'"
1392,1392,연예,"백서빈,'깔끔 청년'"


In [129]:
for i in range(1394):
    nid, cid, tt = news.loc[i]
    tt = tt.replace("'", '"')
    sql = "INSERT INTO test_news(news_id, cate_id, title) VALUES ('" + str(nid) + "', '" + str(cid) + "', '" + tt + "')"
    print(sql)
    cursor.execute(sql)
    con.commit()

INSERT INTO test_news(news_id, cate_id, title) VALUES ('0', '정치', '이탈리아 지지표 흡수는 기본… 중립 성향 표심 잡는 게 관건 ')
INSERT INTO test_news(news_id, cate_id, title) VALUES ('1', '정치', '이준석 “영남 정치인들 편하게 놔두지 않겠다”…영남 신당 추진 시사')
INSERT INTO test_news(news_id, cate_id, title) VALUES ('2', '정치', '주호영 “서울 갈 일 없다”…이준석 “대구에서 어려운 승부할 것”')
INSERT INTO test_news(news_id, cate_id, title) VALUES ('3', '정치', '28억 재산 누락" 김대기 "단순 실수"…야당 "대국민 사과해야"')
INSERT INTO test_news(news_id, cate_id, title) VALUES ('4', '정치', '與 핵심, "거취 압박" 대응 자제...내부선 불만 기류도')
INSERT INTO test_news(news_id, cate_id, title) VALUES ('5', '정치', '"왜 "안철수씨"라고 했냐면"…이준석 "복국집 고함" 입 열었다')
INSERT INTO test_news(news_id, cate_id, title) VALUES ('6', '정치', ' 한동훈 "마약으로 어떻게 국면을 전환하겠다는 것인지 오히려 묻고 싶다"')
INSERT INTO test_news(news_id, cate_id, title) VALUES ('7', '정치', '민주당 내부 "도덕성 회복해야" "이재명부터 험지로" 목소리 나와')
INSERT INTO test_news(news_id, cate_id, title) VALUES ('8', '정치', '홍준표 “듣보잡이 설친다” / 친명 “총선 위기감” / 신원식, 주식 했다 혼쭐')
INSERT INTO test_news(news_id, cate_id, 